# Technical Documentation

## Introduction

Ce document présente l'analyse technique des sources de données acquises dans le cadre du projet ainsi que les choix d'implémentation réalisés pour leur exploitation.

Il documente :

- la structure des données acquises ;
- les contraintes techniques rencontrées ;
- les choix d'implémentation retenus ;
- les datasets produits ;
- les scripts développés pour préparer les phases suivantes du projet.

---

# SRC-001 - Benchmark Dataset

## Source Analysis

Le fichier benchmark est composé des worksheets suivantes :

```text
Disclaimers
Holdings
Historical
Performance
Distributions
```

---

### Worksheet : Disclaimers

#### Contenu

- mentions légales ;
- informations réglementaires ;
- conditions d'utilisation ;
- informations relatives à BlackRock.

#### Décision

```text
Exclure du périmètre analytique
```

---

### Worksheet : Holdings

#### Contenu

Composition détaillée du portefeuille ainsi que les principales informations descriptives du fonds.

#### Champs observés

```text
Ticker
Name
Sector
Asset Class
Market Value
Weight (%)
Notional Value
Quantity
Price
Location
Exchange
Currency
FX Rate
Accrual Date
```

#### Décision

```text
Conserver
```

---

### Worksheet : Historical

#### Contenu

Historique quotidien du fonds.

#### Champs observés

```text
As Of
NAV per Share
Ex-Dividends
Shares Outstanding
Non-FV NAV
```

#### Décision

```text
Conserver
```

---

### Worksheet : Performance

#### Contenu

Historique des performances mensuelles du fonds depuis sa création.

#### Décision

```text
Conserver
```

---

### Worksheet : Distributions

#### Contenu

Historique des distributions versées par le fonds.

#### Champs observés

```text
Record Date
Ex-Date
Payable Date
Total Distribution
Income
ST Cap Gains
LT Cap Gains
Return of Capital
```

#### Décision

```text
Conserver
```

---

## Technical Implementation

### extract_benchmark.py

#### Objectif

Télécharger automatiquement le benchmark depuis la plateforme iShares et conserver le fichier dans sa forme d'origine au sein de la couche RAW.

#### Dataset produit

```text
data/01_raw/benchmark/benchmark_holdings.xls
```

#### Observation technique

Bien que le fichier téléchargé possède l'extension :

```text
.xls
```

il ne correspond pas à un classeur Excel classique.

L'analyse du contenu a montré que le benchmark est distribué sous la forme d'un document SpreadsheetML XML contenant plusieurs worksheets.

Cette caractéristique a nécessité une stratégie d'extraction spécifique pour les traitements ultérieurs.

---

### securities_benchmark_candidates.py

#### Objectif

Construire l'univers de titres utilisé comme entrée de l'acquisition du Securities Dataset (SRC-002).

Le script extrait les instruments financiers présents dans la worksheet Holdings afin de préparer leur enrichissement via OpenFIGI.

#### Entrée

```text
data/01_raw/benchmark/benchmark_holdings.xls
```

#### Dataset produit

```text
data/01_raw/securities/security_candidates.csv
```

---

#### Approche initiale

Une première implémentation reposant sur :

```python
pandas.read_excel()
```

a été testée.

Cette approche a échoué car le fichier benchmark n'est pas un classeur Excel moderne compatible avec OpenPyXL.

---

#### Approche alternative

Une seconde implémentation basée sur :

```python
xml.etree.ElementTree
```

a également été testée.

Le document contenait plusieurs éléments incompatibles avec un parsing XML strict, empêchant son exploitation directe.

---

#### Solution retenue

Le benchmark a finalement été traité comme un document texte.

Le processus d'extraction est composé des étapes suivantes :

1. lecture du contenu complet du fichier ;
2. localisation de la worksheet Holdings à l'aide d'expressions régulières ;
3. extraction de l'ensemble des cellules de données ;
4. détection dynamique de la ligne d'entête ;
5. reconstruction tabulaire des enregistrements dans un DataFrame pandas.

Cette approche permet de traiter la structure réelle du benchmark sans dépendre d'un moteur Excel ou d'un parseur XML strict.

---

#### Construction de l'univers de titres

L'analyse de la worksheet Holdings a permis d'identifier plusieurs catégories d'actifs :

```text
Equity
Cash
FX
Futures
Money Market
Cash Collateral and Margins
```

Le futur Security Master ayant vocation à référencer les instruments financiers investissables, seules les positions Equity ont été retenues.

#### Règle appliquée

Conserver :

```text
Asset Class = Equity
```

Exclure :

```text
Cash
FX
Futures
Money Market
Cash Collateral and Margins
```

---

#### Résultats obtenus

La reconstruction de la worksheet Holdings a permis d'obtenir :

```text
180 lignes
14 colonnes
```

Après application des règles de filtrage :

```text
120 titres Equity
120 titres uniques
```

Ce résultat est cohérent avec l'information fournie dans les métadonnées du benchmark :

```text
Number of Securities (excluding cash and derivatives)
=
120
```

---

#### Colonnes conservées

```text
Ticker
Name
Location
Exchange
Currency
Asset Class
```

---

#### Utilisation future

Le dataset produit constitue l'entrée du script :

```text
extract_securities.py
```

qui sera chargé d'interroger l'API OpenFIGI afin de construire le Security Master du projet.

Les enrichissements attendus comprennent notamment :

```text
FIGI
Composite FIGI
Share Class FIGI
ISIN
Security Type
Market Sector
Exchange Code
Security Description
```

---

## Summary

### Worksheets retenues

```text
Holdings
Historical
Performance
Distributions
```

### Worksheets exclues

```text
Disclaimers
```

### Datasets produits

```text
data/01_raw/benchmark/benchmark_holdings.xls

data/01_raw/securities/security_candidates.csv
```

### Scripts implémentés

```text
extract_benchmark.py

securities_benchmark_candidates.py
```

---

## Conclusion

L'analyse du Benchmark Dataset a permis d'identifier les données pertinentes pour les futures phases d'investissement, de suivi de performance et de contrôle qualité.

Les scripts développés dans cette phase ont permis :

- l'acquisition automatisée du benchmark ;
- l'analyse de sa structure interne ;
- l'identification des titres investissables ;
- la construction d'un univers de 120 instruments financiers destiné à l'enrichissement du Securities Dataset (SRC-002).

Le dataset `security_candidates.csv` constitue désormais la base de travail pour la construction du Security Master et l'intégration de la source OpenFIGI.

In [1]:
with open(".env", "w") as f:
    f.write("OPENFIGI_API_KEY=TA_CLE_ICI")


In [2]:
from pathlib import Path

Path(".env").exists()


True

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("OPENFIGI_API_KEY"))

TA_CLE_ICI


In [5]:
from pathlib import Path

print(Path.cwd())

C:\Users\MameBARBOSAMONTEIRO\projet01\documentation


In [6]:
from pathlib import Path

print(Path(".env").resolve())

C:\Users\MameBARBOSAMONTEIRO\projet01\documentation\.env


In [10]:
from pathlib import Path
import shutil

shutil.move(".env", "../.env")

'../.env'